# The flux analyses under everything else

`strain_design` and `optforce` are search procedures built on a handful of
primitives, and a design is only as trustworthy as what those primitives report.
This notebook covers them directly: the alternate optima FBA hides, the deletion
scans, the yield-versus-growth trade-off, the enzyme budget, and what happens
when more than one organism shares a medium.

Running theme: **FBA returns one optimal flux vector out of many, and the others
are not usually reported.** Most of what follows is a way of asking what else was
possible.

In [1]:
import omicverse as ov
ov.plot_set()

/scratch/users/steorra/env/omicdev/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)


/scratch/users/steorra/env/omicdev/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


🔬 Starting plot initialization...
🧬 Detecting GPU devices…
✅ NVIDIA CUDA GPUs detected: 1
    • [CUDA 0] NVIDIA H100 80GB HBM3
      Memory: 79.1 GB | Compute: 9.0

   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

🔖 Version: 2.2.1rc1   📚 Tutorials: https://omicverse.readthedocs.io/
✅ plot_set complete.



In [2]:
model = ov.synbio.load_gem('textbook')
solution = ov.synbio.fba(model)
print(f'{len(model.reactions)} reactions, {len(model.genes)} genes')
print(f'wild-type growth {solution.objective_value:.4f} /h')

INFO:cobra.io.web.load:Attempting to fetch 'textbook' from the Cobrapy repository.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


95 reactions, 137 genes
wild-type growth 0.8739 /h


## 1 — Which of the optima did FBA hand back?

The LP has a unique *objective value* and usually many flux vectors that achieve
it. pFBA picks among them by minimising total flux, on the argument that a cell
does not run reactions it does not need.

In [3]:
parsimonious = ov.synbio.pfba(model)
print(f'FBA  total |flux| {sum(abs(v) for v in solution.fluxes):.1f}')
print(f'pFBA total |flux| {sum(abs(v) for v in parsimonious.fluxes):.1f}')
print(f'growth unchanged: {parsimonious.objective_value:.4f}')

FBA  total |flux| 518.4
pFBA total |flux| 518.4
growth unchanged: 518.4221


Same growth, less flux. The gap is the size of the arbitrary part of the FBA
answer.

FVA quantifies it per reaction: the range each flux could take while the objective
stays at its optimum.

In [4]:
ranges = ov.synbio.fva(model, fraction_of_optimum=1.0)
width = (ranges['maximum'] - ranges['minimum']).sort_values(ascending=False)
print('the widest-open reactions at the optimum:')
print(width.head(6).round(2).to_string())

the widest-open reactions at the optimum:
FRD7           994.94
SUCDi          994.94
EX_gln__L_e      0.00
EX_mal__L_e      0.00
FRUpts2          0.00
GLNabc           0.00


In [5]:
print(f'{int((width > 1e-6).sum())} of {len(width)} reactions have any freedom at all')
print(f'{int((width > 100).sum())} can vary by more than 100 mmol/gDW/h')

2 of 95 reactions have any freedom at all
2 can vary by more than 100 mmol/gDW/h


A reaction with a range in the thousands is in a loop: it carries flux around a
cycle that produces and consumes nothing net, so the LP is indifferent to its
value. FRD7 and SUCDi in this model are the textbook case, and
`validate_gem(check_cycles=True)` reports them by name.

Relaxing the objective is what a real design question looks like — "what can this
reaction do while the cell still grows at 90% of maximum":

In [6]:
relaxed = ov.synbio.fva(model, reaction_list=['EX_succ_e', 'EX_ac_e', 'EX_etoh_e'],
                        fraction_of_optimum=0.9)
print(relaxed.round(3).to_string())

           minimum  maximum
EX_succ_e      0.0    1.674
EX_ac_e        0.0    3.814
EX_etoh_e      0.0    2.214


## 2 — Deletion scans

Which single genes matter, and which pairs matter only together.

In [7]:
singles = ov.synbio.single_gene_deletion(model)
print(f'{len(singles)} genes scanned, {int((singles["growth"] < 1e-6).sum())} essential')
print(singles.nsmallest(6, 'growth').round(4).to_string())

137 genes scanned, 5 essential
         ids  growth   status
25   {b1136} -0.0000  optimal
116  {b0720} -0.0000  optimal
11   {b1779}  0.0000  optimal
4    {b2926}  0.0000  optimal
65   {b2779}  0.0000  optimal
75   {s0001}  0.2111  optimal


In [8]:
subset = [g.id for g in list(model.genes)[:12]]
doubles = ov.synbio.double_gene_deletion(model, gene_list=subset)
print(f'{len(doubles)} pairs scanned from {len(subset)} genes')
print(doubles.nsmallest(4, 'growth').round(4).to_string())

78 pairs scanned from 12 genes
               ids  growth   status
36  {b0118, b1276} -0.0000  optimal
9   {b0116, s0001}  0.2108  optimal
40         {s0001}  0.2111  optimal
10  {s0001, b0727}  0.2111  optimal


A double scan is quadratic, so it is run on a subset here. The pairs worth the
compute are the ones where both singles are viable and the double is not —
synthetic lethality — because those are exactly the interactions a single scan
cannot see.

**A caution that applies to every row above.** These are FBA predictions, and FBA
is systematically optimistic about knockouts: it re-optimises the whole network as
though the cell had already evolved a new optimum. `knockout_flux(method='moma')`
and `method='room'` ask instead for the nearest feasible state to the wild type.
They disagree with FBA often, and the disagreement is the useful part.

## 3 — The trade-off you are designing against

Growth and product compete for the same carbon. A production envelope draws the
whole feasible region rather than one point on it.

In [9]:
anaerobic = ov.synbio.load_gem('textbook')
anaerobic.reactions.EX_o2_e.lower_bound = 0.0
envelope = ov.synbio.production_envelope(anaerobic, 'EX_succ_e')
print(envelope[['EX_succ_e', 'flux_minimum', 'flux_maximum']].round(4).head(5).to_string(index=False))

INFO:cobra.io.web.load:Attempting to fetch 'textbook' from the Cobrapy repository.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


INFO:cobra.medium.boundary_types:Compartment `e` sounds like an external compartment. Using this one without counting boundary reactions.


 EX_succ_e  flux_minimum  flux_maximum
    0.0000           0.0        0.2117
    0.7319           0.0        0.2036
    1.4638           0.0        0.1955
    2.1956           0.0        0.1873
    2.9275           0.0        0.1792


In [10]:
print(f'succinate spans {envelope["EX_succ_e"].min():.2f} to {envelope["EX_succ_e"].max():.2f} mmol/gDW/h')
print(f'max growth falls {envelope["flux_maximum"].max():.4f} -> {envelope["flux_maximum"].min():.4f} /h')
print(f'min growth across the whole envelope: {envelope["flux_minimum"].max():.4f} /h')

succinate spans 0.00 to 13.91 mmol/gDW/h
max growth falls 0.2117 -> 0.0000 /h
min growth across the whole envelope: 0.0000 /h


Read the axes carefully. The grid runs over **succinate**, and `flux_minimum` /
`flux_maximum` are the range of the **objective — growth** — reachable at each
succinate level. The table answers "if I force this much product out, how fast can
the cell still grow", and the answer falls monotonically to zero. That is the
trade-off, drawn rather than asserted.

What it does *not* show is coupling. `flux_minimum` is 0 everywhere, because at
any succinate level there is always a feasible state with no growth. The design
question is the transpose — **at high growth, what is the least succinate the cell
must make?** If that is zero then nothing is coupled: the cell is free to grow and
make nothing, and it will.

In [11]:
with anaerobic as constrained:
    biomass = [r for r in constrained.reactions if r.objective_coefficient][0]
    biomass.lower_bound = 0.99 * ov.synbio.fba(constrained).objective_value
    constrained.objective = 'EX_succ_e'
    constrained.objective.direction = 'min'
    print(f'guaranteed succinate at 99% of max growth: '
          f'{ov.synbio.fba(constrained).objective_value:.4f} mmol/gDW/h')

guaranteed succinate at 99% of max growth: 0.0000 mmol/gDW/h


Small but not zero — carbon has to go somewhere anaerobically. A real
growth-coupled design pushes that number up; a design that leaves it at zero has
not coupled anything, whatever its production envelope looks like.

## 4 — The enzyme budget

FBA has no notion that a flux needs an enzyme to carry it. `ec_model` adds one.

In [12]:
constrained = ov.synbio.ec_model(ov.synbio.load_gem('textbook'), {'PFK': 25.0}, total_protein=0.2)
print(f'growth under a 0.2 g/gDW enzyme budget: {ov.synbio.fba(constrained).objective_value:.4f} /h')
print(f'unconstrained was {solution.objective_value:.4f} /h')

INFO:cobra.io.web.load:Attempting to fetch 'textbook' from the Cobrapy repository.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


growth under a 0.2 g/gDW enzyme budget: 0.8739 /h
unconstrained was 0.8739 /h


Comparing enzyme variants requires holding the budget fixed, which is what
`apply_kcat` is for: it reuses the budget already on the model instead of
re-deriving one from the kcat under test.

In [13]:
for kcat in (0.4, 4.0, 40.0, 400.0):
    variant = ov.synbio.apply_kcat(constrained.copy(), {'PFK': kcat})
    print(f'  kcat(PFK) = {kcat:6.1f} /s  ->  growth {ov.synbio.fba(variant).objective_value:.4f} /h')

INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


  kcat(PFK) =    0.4 /s  ->  growth 0.6347 /h


...

  kcat(PFK) =    4.0 /s  ->  growth 0.8739 /h


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


  kcat(PFK) =   40.0 /s  ->  growth 0.8739 /h


  kcat(PFK) =  400.0 /s  ->  growth 0.8739 /h


Monotonic, as physics requires: a faster enzyme cannot make the cell grow slower
under the same protein budget.

Worth stating because it was not always true. Auto-sizing the budget from the kcat
map under test inverted it — enzyme mass demand goes as 1/kcat, so a slower enzyme
bought a *bigger* budget, and an 830× slower PFK predicted 2× more growth.
`apply_kcat` was also unreachable, raising on any model that already carried a
budget, which is every model it was meant for.

`ec_model` will also tell you when a kcat is not believable:

In [14]:
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    ov.synbio.ec_model(ov.synbio.load_gem('textbook'), {'PFK': 0.04})
print([str(w.message)[:150] for w in caught if '不可能多的酶' in str(w.message)][0])

INFO:cobra.io.web.load:Attempting to fetch 'textbook' from the Cobrapy repository.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


以下反应的 kcat 低到需要不可能多的酶(PFK: 2.08 g/gDW) —— 按野生型通量算,单个酶就要占干重的 25% 以上,超过 1.0 则直接不可能。这通常说明 kcat 预测值错了(先去 BRENDA/SABIO-RK 核一下),而不是这个酶真的成了瓶颈。


0.040 /s for phosphofructokinase — a value a sequence-to-kcat predictor will
happily return — needs **2.08 g of PfkA per gram of dry cell weight** at the
wild-type flux. 208% of the cell. The number falsifies itself, and nothing was
checking it.

## 5 — More than one organism

A community shares a medium, and the members compete for it.

In [15]:
aerobe = ov.synbio.load_gem('textbook')
anaerobe = ov.synbio.load_gem('textbook')
anaerobe.reactions.EX_o2_e.lower_bound = 0.0
community = ov.synbio.community_model({'aerobe': aerobe, 'anaerobe': anaerobe})
print(community)

INFO:cobra.io.web.load:Attempting to fetch 'textbook' from the Cobrapy repository.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


INFO:cobra.io.web.load:Attempting to fetch 'textbook' from the Cobrapy repository.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


community


In [16]:
result = ov.synbio.steadycom(community)
print(result)
print('abundances:', {k: round(v, 3) for k, v in result.abundances.items()})

CommunityResult(steadycom, 2 members, community growth=0.5942, 1 cross-feeding links)
abundances: {'aerobe': 1.0, 'anaerobe': -0.0}


`micom_grow` dispatches to MICOM when installed — the same problem with a
different trade-off between community and individual growth rates — and
`me_model` is the entry point for a full ME-model backend. Both are optional, and
both report a missing dependency as a message rather than an import error at the
top of a script.

In [17]:
for name, call in (('micom_grow', lambda: ov.synbio.micom_grow(community)),
                   ('me_model', lambda: ov.synbio.me_model())):
    try:
        call()
        print(f'{name}: available')
    except Exception as exc:
        print(f'{name}: {type(exc).__name__} — {str(exc).splitlines()[0][:120]}')

micom_grow: ImportError — method 需要 MICOM(pip install micom)。MICOM 用协作权衡目标,与内置的 steadycom(严格等速率)是不同的建模假设 —— 两者都跑一遍再比较是有意义的。
me_model: ImportError — ME 模型需要 COBRAme(github.com/SBRG/cobrame,无 PyPI 包)以及一份物种特异的 ME 重建(如 ecolime 的 iJL1678b),还需要高精度求解器(qMINOS 或 SoPlex)——ME 模型


## 6 — Where this layer stops

`enzyme_dynamics` is worth naming here so it is not looked for in the wrong place:
it runs **molecular dynamics** on a protein structure, not kinetics on a flux
model. Constraint-based analysis has no time axis at all — every result above is a
statement about steady states.

For a time axis on the *metabolic* side, `dynamic_fba` integrates substrate
depletion and product accumulation over a batch. For the *protein* side,
`enzyme_dynamics` is the entry point.

In [18]:
try:
    ov.synbio.enzyme_dynamics(ov.synbio.reference_protein('gfp')[:40], ns=0.001)
except Exception as exc:
    print(f'{type(exc).__name__}: {str(exc).splitlines()[0][:170]}')

[ov.synbio.predict_structure] device=cuda (NVIDIA H100 80GB HBM3, 79 GB) len=40


Some weights of EsmForProteinFolding were not initialized from the model checkpoint at facebook/esmfold_v1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[ov.synbio.predict_structure] done: mean pLDDT=79.0
🧬 ov.mol.simulate — 0.001 ns, output in md_out/
   fixing structure (folded.pdb) at pH 7.0


   solvating: tip3p box, padding 1.0 nm, 0.15 M Na+/Cl-


   system ready: 45352 particles (explicit solvent)
   OpenMM platform: CUDA (precision=mixed)


...

350	0.7000000000000005	-634946.040210424	300.8280784973098	456.7018447974995	1.0002382706785078	475	0:00


400	0.8000000000000006	-635729.6689677914	302.5850405788332	456.7018447974995	1.0002382706785078	472	0:00


450	0.9000000000000007	-635130.7986024972	300.8487616792813	457.05675353197535	0.9994615765457165	475	0:00


500	1.0000000000000007	-635078.0506901899	300.22351969128573	457.05675353197535	0.9994615765457165	455	0:00


   done: 0 frames on platform CUDA


## 7 — Which backends do you actually have?

Nearly every analysis in `ov.synbio` dispatches on a `method=` argument, and most
of the alternatives are optional dependencies. Finding out which are installed by
running into an ImportError halfway through a script is the wrong time.

In [19]:
backends = {
    'reconstruct_gem': ('homology', 'carveme', 'gapseq', 'modelseed'),
    'gapfill_model': ('lp', 'cobra'),
    'contextualize_gem': ('gimme', 'riptide', 'imat', 'init', 'tinit'),
    'strain_design': ('fseof', 'optknock', 'robustknock', 'mcs'),
    'knockout_flux': ('fba', 'moma', 'room'),
}
for function, methods in backends.items():
    print(f'  {function:20s} {", ".join(methods)}')

  reconstruct_gem      homology, carveme, gapseq, modelseed
  gapfill_model        lp, cobra
  contextualize_gem    gimme, riptide, imat, init, tinit
  strain_design        fseof, optknock, robustknock, mcs
  knockout_flux        fba, moma, room


In [20]:
more = {
    'reaction_dg': ('baseline', 'equilibrator'),
    'thermo_fba': ('directionality', 'tmfa'),
    'enzyme_kcat': ('baseline', 'dlkcat', 'unikp'),
    'rbs_strength': ('thermodynamic', 'ostir', 'salis'),
    'simulate_circuit': ('ode', 'ssa', 'gillespie', 'stochastic'),
    'ancestral_reconstruction': ('ml', 'parsimony'),
    'stability_ddg': ('proxy', 'thermompnn'),
}
for function, methods in more.items():
    print(f'  {function:24s} {", ".join(methods)}')

  reaction_dg              baseline, equilibrator
  thermo_fba               directionality, tmfa
  enzyme_kcat              baseline, dlkcat, unikp
  rbs_strength             thermodynamic, ostir, salis
  simulate_circuit         ode, ssa, gillespie, stochastic
  ancestral_reconstruction ml, parsimony
  stability_ddg            proxy, thermompnn


The manufacturability predictors follow the same pattern, and their alternatives
are all **external tools** rather than Python packages: `predict_solubility` takes
`proteinsol`, `aggregation_propensity` takes `aggrescan` or `tango`,
`predict_signal_peptide` takes `signalp`, and `predict_localization` takes
`deeploc`. Each falls back to the built-in heuristic and says so, rather than
failing.

In [21]:
protein = ov.synbio.reference_protein('gfp')
for function, method in (('predict_solubility', 'proteinsol'),
                         ('aggregation_propensity', 'tango'),
                         ('predict_signal_peptide', 'signalp'),
                         ('predict_localization', 'deeploc')):
    try:
        getattr(ov.synbio, function)(protein, method=method)
        print(f'  {function:24s} {method:12s} available')
    except Exception as exc:
        print(f'  {function:24s} {method:12s} {type(exc).__name__}: {str(exc).splitlines()[0][:70]}')

  predict_solubility       proteinsol   ImportError: method='proteinsol' 需要本地可用的 Protein-Sol(Hebditch 2017)。它以网页服务和可下载脚本形式发
  aggregation_propensity   tango        ImportError: method='tango' 需要外部预测器:AGGRESCAN(bioinf.uab.es/aggrescan)与 TANGO(tango
  predict_signal_peptide   signalp      ImportError: method='signalp' 需要 SignalP 6.0(services.healthtech.dtu.dk,学术许可后可本地安装 
  predict_localization     deeploc      ImportError: method='deeploc' 需要 DeepLoc 2.0(services.healthtech.dtu.dk,学术许可)。默认 me


Every one of those falls back or reports; none of them silently substitutes a
different计算 and calls it the same thing. That distinction is what makes a
`method=` argument safe to write in a pipeline.

## What to carry away

**FBA gives you one answer out of many.** pFBA picks a defensible one; FVA tells
you how much was arbitrary. A reaction whose FVA range spans thousands is in a
loop, not doing something interesting.

**A production envelope is not a coupling test.** It shows how fast the cell can
grow at each product level. Coupling is the transpose — the *minimum* product at
high growth — and it has to be asked for separately.

**Hold the budget fixed when comparing enzymes.** `apply_kcat`, not a fresh
`ec_model` per variant; the second re-derives the budget from the very number you
are varying.

**A kcat implies an enzyme mass.** Compute it before believing a prediction:
0.04 /s for a central glycolytic enzyme is 208% of dry weight, and no model that
takes it seriously is worth reading.

**None of this has a time axis.** Constraint-based analysis answers questions
about steady states. When the question is "when", the answer is elsewhere.